In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("ServerLogAnalysis").master("local[*]").getOrCreate()

sc = spark.sparkContext



In [8]:
logs_rdd = sc.textFile("server_logs.txt")

print(logs_rdd.take(5))

['', 'timestamp,server,endpoint,status_code,response_time_ms', '2026-10-20T09:00:01,server01,/login,200,120', '2026-10-20T09:00:05,server01,/products,200,180', '2026-10-20T09:00:10,server02,/checkout,500,900']


In [9]:
parsed_rdd = logs_rdd.map(
    lambda line: [field.strip() for field in line.split(",")]
).filter(
    lambda fields: len(fields) == 5
    and fields[3].isdigit()
    and fields[4].isdigit()
).map(
    lambda fields: (
        fields[0],
        fields[1],
        fields[2],
        int(fields[3]),
        int(fields[4])
    )
)

print("Total valid log records:", parsed_rdd.count())
print(parsed_rdd.take(3))

Total valid log records: 15
[('2026-10-20T09:00:01', 'server01', '/login', 200, 120), ('2026-10-20T09:00:05', 'server01', '/products', 200, 180), ('2026-10-20T09:00:10', 'server02', '/checkout', 500, 900)]


In [10]:
failed_rdd = parsed_rdd.filter(
    lambda record: record[3] >= 400
)

print("Failed requests:")
for record in failed_rdd.collect():
    print(record)

Failed requests:
('2026-10-20T09:00:10', 'server02', '/checkout', 500, 900)
('2026-10-20T09:00:15', 'server01', '/login', 401, 250)
('2026-10-20T09:00:25', 'server02', '/checkout', 503, 1200)
('2026-10-20T09:00:35', 'server01', '/checkout', 404, 400)
('2026-10-20T09:00:45', 'server03', '/search', 500, 800)
('2026-10-20T09:01:00', 'server03', '/login', 403, 300)
('2026-10-20T09:01:10', 'server01', '/products', 500, 750)


In [11]:
errors_by_status = failed_rdd.map(
    lambda record: (record[3], 1)
).reduceByKey(
    lambda a, b: a + b
)

print("Errors by status code:")
for status, count in sorted(errors_by_status.collect()):
    print(status, count)

Errors by status code:
401 1
403 1
404 1
500 3
503 1


In [12]:
requests_by_endpoint = parsed_rdd.map(
    lambda record: (record[2], 1)
).reduceByKey(
    lambda a, b: a + b
)

print("Requests by endpoint:")
for endpoint, count in sorted(requests_by_endpoint.collect()):
    print(endpoint, count)

Requests by endpoint:
/checkout 4
/login 4
/products 4
/search 3


In [13]:
server_response = parsed_rdd.map(
    lambda record: (record[1], (record[4], 1))
).reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda value: round(value[0] / value[1], 2)
)

print("Average response time by server:")
for server, avg_time in sorted(server_response.collect()):
    print(server, avg_time, "ms")

Average response time by server:
server01 341.67 ms
server02 604.0 ms
server03 512.5 ms


In [14]:
failed_by_server = failed_rdd.map(
    lambda record: (record[1], 1)
).reduceByKey(
    lambda a, b: a + b
)

failed_counts = failed_by_server.collect()

print("Failures by server:")
for server, count in sorted(failed_counts):
    print(server, count)

if failed_counts:
    max_failures = max(count for server, count in failed_counts)

    top_servers = [
        (server, count)
        for server, count in failed_counts
        if count == max_failures
    ]

    print("Server(s) with most failures:", top_servers)

Failures by server:
server01 3
server02 2
server03 2
Server(s) with most failures: [('server01', 3)]


In [15]:
failed_rdd = parsed_rdd.filter(
    lambda record: record[3] >= 400
)

errors_by_status = failed_rdd.map(
    lambda record: (record[3], 1)
).reduceByKey(
    lambda a, b: a + b
)

print(errors_by_status.collect())

[(500, 3), (404, 1), (401, 1), (503, 1), (403, 1)]


In [16]:
output_rdd = errors_by_status.map(
    lambda item: f"HTTP {item[0]}: {item[1]} errors"
)

output_rdd.saveAsTextFile("server_error_summary")

print("Error summary saved successfully!")

Error summary saved successfully!


In [18]:
endpoint_response = parsed_rdd.map(
    lambda record: (record[2], (record[4], 1))
).reduceByKey(
    lambda a, b: (a[0] + b[0], a[1] + b[1])
).mapValues(
    lambda value: round(value[0] / value[1], 2)
)

slow_endpoints = endpoint_response.filter(
    lambda item: item[1] > 500
)

print("Endpoints with average response time > 500 ms:")
for endpoint, avg_time in sorted(slow_endpoints.collect()):
    print(endpoint, avg_time, "ms")

Endpoints with average response time > 500 ms:
/checkout 712.5 ms
/search 666.67 ms
